# Naive Bayes

A probabilistic classifier based on Bayes' theorem with the "naive" assumption of feature independence.

$$P(y|x_1,...,x_n) \propto P(y) \prod_{i=1}^{n} P(x_i|y)$$

Despite the strong independence assumption, Naive Bayes works surprisingly well for text classification.

1. **Gaussian Naive Bayes** - For continuous features
2. **Multinomial Naive Bayes** - For text/count data (TF-IDF)
3. **Text Classification Pipeline** - End-to-end newsgroup classification

**Dataset**: 20 Newsgroups (text classification)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

sns.set_theme(style="whitegrid")

In [ ]:
# Load a subset of newsgroups
categories = ["sci.med", "sci.space", "rec.sport.baseball", "talk.politics.guns"]

train_data = fetch_20newsgroups(subset="train", categories=categories, remove=("headers", "footers", "quotes"))
test_data = fetch_20newsgroups(subset="test", categories=categories, remove=("headers", "footers", "quotes"))

print(f"Training samples: {len(train_data.data)}")
print(f"Test samples: {len(test_data.data)}")
print(f"Categories: {train_data.target_names}")
print(f"\nSample document:\n{train_data.data[0][:300]}...")

In [ ]:
# Text classification pipeline: TF-IDF + Multinomial NB
text_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=10000, stop_words="english", ngram_range=(1, 2))),
    ("clf", MultinomialNB(alpha=0.1)),  # alpha = Laplace smoothing
])

text_pipe.fit(train_data.data, train_data.target)
y_pred = text_pipe.predict(test_data.data)

print(f"Accuracy: {text_pipe.score(test_data.data, test_data.target):.3f}\n")
print(classification_report(test_data.target, y_pred, target_names=train_data.target_names))

In [ ]:
# Effect of smoothing parameter alpha
alphas = np.logspace(-3, 1, 20)
scores = []
for alpha in alphas:
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=10000, stop_words="english")),
        ("clf", MultinomialNB(alpha=alpha)),
    ])
    cv_score = cross_val_score(pipe, train_data.data, train_data.target, cv=5, scoring="accuracy")
    scores.append(cv_score.mean())

plt.figure(figsize=(8, 5))
plt.semilogx(alphas, scores, "o-", color="teal")
plt.xlabel("Alpha (smoothing)")
plt.ylabel("CV Accuracy")
plt.title("Multinomial NB: Effect of Laplace Smoothing")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Most informative features per class
feature_names = text_pipe.named_steps["tfidf"].get_feature_names_out()
log_probs = text_pipe.named_steps["clf"].feature_log_prob_

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, i in zip(axes.flat, range(len(categories))):
    top_idx = np.argsort(log_probs[i])[-15:]
    ax.barh(range(15), log_probs[i][top_idx], color="teal")
    ax.set_yticks(range(15))
    ax.set_yticklabels(feature_names[top_idx])
    ax.set_title(f"Top words: {categories[i]}")

plt.tight_layout()
plt.show()

## Key Takeaways

1. **Naive Bayes excels at text classification** - fast training, good baseline
2. **MultinomialNB for TF-IDF/count data**, GaussianNB for continuous features
3. **Laplace smoothing (alpha)** prevents zero probabilities for unseen features
4. **Very fast** - scales linearly with number of features and samples
5. **Calibration issue** - predicted probabilities are often extreme (close to 0 or 1); use CalibratedClassifierCV if you need calibrated probabilities